# Incremental Orders Fact Pipeline - Python/Pandas
## Bronze → Silver → Gold → Parent Monthly

**S3 Landing**: `s3://sportx-bar/orders/landing/*.csv`  
**Unity Catalog credentials** work automatically via Spark  
**Output**: All CSV files in `data/` folder  
**Incremental**: Processes new files, merges with existing data

Run cells sequentially from top to bottom.

**Import Required Libraries**

In [0]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from dateutil import parser

print("✓ Core libraries imported")

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
%run /Workspace/data_project/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
# Define parameters
catalog = "fmcg"
data_source = "orders"

# S3 paths (using Unity Catalog credential)
s3_bucket = "sportx-bar"
base_path = f's3://{s3_bucket}/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"

print(f"Catalog: {catalog}")
print(f"Data source: {data_source}")
print(f"Landing: {landing_path}")
print(f"Processed: {processed_path}")

# Create data directory for outputs
os.makedirs("data", exist_ok=True)
print("\n✓ Data directory ready")

## Bronze

In [0]:
# Read NEW incremental data from S3 landing zone
# Using Spark (UC credentials) then convert to pandas

print(f"Reading from: {landing_path}")

try:
    # Read CSV from S3 using Spark
    df_spark = spark.read.csv(f"{landing_path}/*.csv", header=True, inferSchema=True)
    
    # Convert to pandas
    df = df_spark.toPandas()
    
    # Add Bronze metadata
    df["read_timestamp"] = pd.Timestamp.now()
    df["file_name"] = f"{data_source}.csv"  # Simplified - all files treated as batch
    df["file_size"] = None
    
    print(f"\n✓ Loaded {len(df)} NEW rows from landing zone")
    print(f"\nColumns: {list(df.columns)}")
    
except Exception as e:
    print(f"\n⚠️  No files found in landing zone: {landing_path}")
    print(f"\nThis is EXPECTED for incremental loads when there's no new data.")
    print(f"\nTo test the pipeline:")
    print(f"  1. Upload CSV files to: {landing_path}")
    print(f"  2. Files should contain: order_id, order_placement_date, customer_id, product_id, order_qty")
    print(f"  3. Re-run this notebook to process the new files")
    print(f"\nPipeline is ready and waiting for new data! ✓")
    df = None  # No data to process

In [0]:
# Check if there's data to process
if df is None:
    print("\n⚠️  No new data to process. Stopping here.")
    print("\nThe pipeline is ready and will run when new files arrive in the landing zone.")
    dbutils.notebook.exit("No new data to process")

# Append new rows to Bronze full history file
bronze_history_path = f"data/bronze_{data_source}.csv"

if os.path.exists(bronze_history_path):
    # Append to existing
    df.to_csv(bronze_history_path, mode='a', header=False, index=False)
    print(f"✓ Appended {len(df)} rows to: {bronze_history_path}")
else:
    # Create new file
    df.to_csv(bronze_history_path, index=False)
    print(f"✓ Created new Bronze history: {bronze_history_path} with {len(df)} rows")

### Staging table to process just the arrived incremenal data

In [0]:
# Save current batch to staging (overwrite - this is just current incremental data)
staging_bronze_path = f"data/staging_bronze_{data_source}.csv"
df.to_csv(staging_bronze_path, index=False)

print(f"✓ Bronze staging saved: {staging_bronze_path} with {len(df)} rows")
print(f"  This contains ONLY the current batch for Silver processing")

### Moving files from source to processed directory

In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

## Silver

In [0]:
# Load Bronze staging (current incremental batch only)
df_orders = pd.read_csv(f"data/staging_bronze_{data_source}.csv")

print(f"Loaded {len(df_orders)} rows from Bronze staging")
print(f"\nFirst 2 rows:")
display(df_orders.head(2))

**Transformations**

In [0]:
# 1. Keep only rows where order_qty is present
print(f"Rows before filter: {len(df_orders)}")
df_orders = df_orders[df_orders['order_qty'].notna()]
print(f"Rows after removing null order_qty: {len(df_orders)}")

# 2. Clean customer_id → keep numeric, else set to 999999
def clean_customer_id(val):
    if pd.isna(val):
        return "999999"
    val_str = str(val)
    if val_str.isdigit():
        return val_str
    return "999999"

df_orders['customer_id'] = df_orders['customer_id'].apply(clean_customer_id)

# 3. Remove weekday name from date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
import re
df_orders['order_placement_date'] = df_orders['order_placement_date'].astype(str).apply(
    lambda x: re.sub(r'^[A-Za-z]+,\s*', '', x)
)

# 4. Parse order_placement_date using multiple formats
def parse_date(date_str):
    if pd.isna(date_str) or date_str == 'None' or date_str == '':
        return None
    
    formats = [
        '%Y/%m/%d',
        '%d-%m-%Y',
        '%d/%m/%Y',
        '%B %d, %Y',  # "July 01, 2025"
    ]
    
    for fmt in formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except:
            continue
    
    # Fallback: try dateutil parser
    try:
        return parser.parse(date_str)
    except:
        return None

df_orders['order_placement_date'] = df_orders['order_placement_date'].apply(parse_date)

# 5. Drop duplicates
print(f"\nRows before dedup: {len(df_orders)}")
df_orders = df_orders.drop_duplicates(
    subset=['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'],
    keep='first'
)
print(f"Rows after dedup: {len(df_orders)}")

# 6. Convert product_id to string
df_orders['product_id'] = df_orders['product_id'].astype(str)

print(f"\n✓ Transformations complete")

In [0]:
# Check date range in current batch
min_date = df_orders['order_placement_date'].min()
max_date = df_orders['order_placement_date'].max()

print(f"\nDate range in current batch:")
print(f"  Min date: {min_date}")
print(f"  Max date: {max_date}")

**Join with products**

In [0]:
# Load products dimension
df_products = pd.read_csv("data/silver_products.csv")
print(f"Loaded {len(df_products)} products")

# Join orders with products to get product_code
df_joined = df_orders.merge(
    df_products[['product_id', 'product_code']],
    on='product_id',
    how='inner'
)

print(f"\n✓ Joined with products: {len(df_joined)} rows")
print(f"\nFirst 5 rows:")
display(df_joined.head(5))

In [0]:
# Merge logic: upsert into Silver full history
# Match keys: order_placement_date, order_id, product_code, customer_id

silver_path = f"data/silver_{data_source}.csv"

if not os.path.exists(silver_path):
    # Create new Silver table
    df_joined.to_csv(silver_path, index=False)
    print(f"✓ Created new Silver table: {silver_path} with {len(df_joined)} rows")
else:
    # Load existing Silver data
    df_silver_existing = pd.read_csv(silver_path)
    print(f"Existing Silver rows: {len(df_silver_existing)}")
    
    # Parse date column if it's stored as string
    df_silver_existing['order_placement_date'] = pd.to_datetime(df_silver_existing['order_placement_date'])
    
    # Create composite key for matching
    df_joined['_merge_key'] = (df_joined['order_placement_date'].astype(str) + '|' + 
                                df_joined['order_id'].astype(str) + '|' +
                                df_joined['product_code'].astype(str) + '|' +
                                df_joined['customer_id'].astype(str))
    
    df_silver_existing['_merge_key'] = (df_silver_existing['order_placement_date'].astype(str) + '|' + 
                                         df_silver_existing['order_id'].astype(str) + '|' +
                                         df_silver_existing['product_code'].astype(str) + '|' +
                                         df_silver_existing['customer_id'].astype(str))
    
    # Remove rows from existing that will be updated
    merge_keys_to_update = df_joined['_merge_key'].unique()
    df_silver_keep = df_silver_existing[~df_silver_existing['_merge_key'].isin(merge_keys_to_update)]
    
    print(f"Rows to keep (not updated): {len(df_silver_keep)}")
    print(f"Rows to upsert (new + updated): {len(df_joined)}")
    
    # Drop the merge key column before concatenating
    df_silver_keep = df_silver_keep.drop(columns=['_merge_key'])
    df_joined = df_joined.drop(columns=['_merge_key'])
    
    # Concatenate
    df_silver_merged = pd.concat([df_silver_keep, df_joined], ignore_index=True)
    
    # Save
    df_silver_merged.to_csv(silver_path, index=False)
    print(f"\n✓ Merged Silver table: {silver_path} now has {len(df_silver_merged)} rows")

### Staging table to process just the arrived incremenal data

In [0]:
# Save Silver staging (current batch only) for Gold processing
staging_silver_path = f"data/staging_silver_{data_source}.csv"
df_joined.to_csv(staging_silver_path, index=False)

print(f"✓ Silver staging saved: {staging_silver_path} with {len(df_joined)} rows")
print(f"  This contains ONLY the current batch for Gold processing")

## Gold

In [0]:
# Load Silver staging and create Gold fact table
df_silver_staging = pd.read_csv(f"data/staging_silver_{data_source}.csv")
print(f"Loaded {len(df_silver_staging)} rows from Silver staging")

# Parse date column
df_silver_staging['order_placement_date'] = pd.to_datetime(df_silver_staging['order_placement_date'])

# Select and rename columns for Gold fact table
df_gold = df_silver_staging[[
    'order_id',
    'order_placement_date',
    'customer_id',
    'product_code',
    'product_id',
    'order_qty'
]].copy()

df_gold = df_gold.rename(columns={
    'order_placement_date': 'date',
    'customer_id': 'customer_code',
    'order_qty': 'sold_quantity'
})

print(f"\nGold fact table preview:")
display(df_gold.head(2))

In [0]:
print(f"Gold fact table rows: {len(df_gold)}")

In [0]:
# Merge into Gold sb_fact_orders (daily level)
# Match keys: date, order_id, product_code, customer_code

gold_sb_path = f"data/gold_sb_fact_{data_source}.csv"

if not os.path.exists(gold_sb_path):
    # Create new Gold table
    df_gold.to_csv(gold_sb_path, index=False)
    print(f"✓ Created new Gold daily fact table: {gold_sb_path} with {len(df_gold)} rows")
else:
    # Load existing Gold data
    df_gold_existing = pd.read_csv(gold_sb_path)
    print(f"Existing Gold daily rows: {len(df_gold_existing)}")
    
    # Parse date column
    df_gold_existing['date'] = pd.to_datetime(df_gold_existing['date'])
    df_gold['date'] = pd.to_datetime(df_gold['date'])
    
    # Create composite key
    df_gold['_merge_key'] = (df_gold['date'].astype(str) + '|' + 
                             df_gold['order_id'].astype(str) + '|' +
                             df_gold['product_code'].astype(str) + '|' +
                             df_gold['customer_code'].astype(str))
    
    df_gold_existing['_merge_key'] = (df_gold_existing['date'].astype(str) + '|' + 
                                       df_gold_existing['order_id'].astype(str) + '|' +
                                       df_gold_existing['product_code'].astype(str) + '|' +
                                       df_gold_existing['customer_code'].astype(str))
    
    # Remove rows that will be updated
    merge_keys = df_gold['_merge_key'].unique()
    df_gold_keep = df_gold_existing[~df_gold_existing['_merge_key'].isin(merge_keys)]
    
    print(f"Rows to keep: {len(df_gold_keep)}")
    print(f"Rows to upsert: {len(df_gold)}")
    
    # Drop merge key
    df_gold_keep = df_gold_keep.drop(columns=['_merge_key'])
    df_gold = df_gold.drop(columns=['_merge_key'])
    
    # Concatenate
    df_gold_merged = pd.concat([df_gold_keep, df_gold], ignore_index=True)
    
    # Save
    df_gold_merged.to_csv(gold_sb_path, index=False)
    print(f"\n✓ Merged Gold daily fact: {gold_sb_path} now has {len(df_gold_merged)} rows")

## Merging with Parent company

- Note: We want data for monthly level but child data is on daily level

**Incremental Load**

In [0]:
# Identify which months are affected by the current batch
# We need to recalculate these months in the parent table

df_child = pd.read_csv(f"data/staging_silver_{data_source}.csv")
df_child['order_placement_date'] = pd.to_datetime(df_child['order_placement_date'])

# Get first day of month for each date
df_child['start_month'] = df_child['order_placement_date'].dt.to_period('M').dt.to_timestamp()

# Get unique months
incremental_months = df_child['start_month'].unique()

print(f"\nMonths affected by current batch: {len(incremental_months)}")
for month in sorted(incremental_months):
    print(f"  {month}")

In [0]:
# Load all daily data for the affected months from Gold sb_fact_orders
# We need ALL days in these months to recalculate the monthly aggregates

df_gold_all = pd.read_csv(f"data/gold_sb_fact_{data_source}.csv")
df_gold_all['date'] = pd.to_datetime(df_gold_all['date'])
df_gold_all['month_start'] = df_gold_all['date'].dt.to_period('M').dt.to_timestamp()

# Filter to affected months only
monthly_table = df_gold_all[df_gold_all['month_start'].isin(incremental_months)]

print(f"\nTotal daily rows for affected months: {len(monthly_table)}")
print(f"\nFirst 10 rows:")
display(monthly_table.head(10))

In [0]:
# Show distinct dates in the affected months
distinct_dates = monthly_table['date'].unique()
print(f"\nDistinct dates: {len(distinct_dates)}")
for date in sorted(distinct_dates)[:20]:  # Show first 20
    print(f"  {date}")
if len(distinct_dates) > 20:
    print(f"  ... and {len(distinct_dates) - 20} more")

In [0]:
# Aggregate daily data to monthly level
# Group by month, product_code, customer_code

df_monthly_recalc = monthly_table.groupby(
    ['month_start', 'product_code', 'customer_code']
).agg({
    'sold_quantity': 'sum'
}).reset_index()

# Rename month_start to date (first day of month)
df_monthly_recalc = df_monthly_recalc.rename(columns={'month_start': 'date'})

print(f"\nMonthly aggregated rows: {len(df_monthly_recalc)}")
print(f"\nFirst 10 rows:")
display(df_monthly_recalc.head(10))

In [0]:
print(f"Monthly fact rows to merge: {len(df_monthly_recalc)}")

In [0]:
# Merge monthly aggregates into parent fact_orders table
# Match keys: date (month), product_code, customer_code

gold_parent_path = f"data/gold_fact_{data_source}.csv"

if not os.path.exists(gold_parent_path):
    # Create new parent table
    df_monthly_recalc.to_csv(gold_parent_path, index=False)
    print(f"✓ Created new parent monthly fact: {gold_parent_path} with {len(df_monthly_recalc)} rows")
else:
    # Load existing parent data
    df_parent_existing = pd.read_csv(gold_parent_path)
    print(f"Existing parent monthly rows: {len(df_parent_existing)}")
    
    # Parse date
    df_parent_existing['date'] = pd.to_datetime(df_parent_existing['date'])
    df_monthly_recalc['date'] = pd.to_datetime(df_monthly_recalc['date'])
    
    # Create composite key
    df_monthly_recalc['_merge_key'] = (df_monthly_recalc['date'].astype(str) + '|' + 
                                        df_monthly_recalc['product_code'].astype(str) + '|' +
                                        df_monthly_recalc['customer_code'].astype(str))
    
    df_parent_existing['_merge_key'] = (df_parent_existing['date'].astype(str) + '|' + 
                                         df_parent_existing['product_code'].astype(str) + '|' +
                                         df_parent_existing['customer_code'].astype(str))
    
    # Remove rows that will be updated (matching months)
    merge_keys = df_monthly_recalc['_merge_key'].unique()
    df_parent_keep = df_parent_existing[~df_parent_existing['_merge_key'].isin(merge_keys)]
    
    print(f"Rows to keep: {len(df_parent_keep)}")
    print(f"Rows to upsert: {len(df_monthly_recalc)}")
    
    # Drop merge key
    df_parent_keep = df_parent_keep.drop(columns=['_merge_key'])
    df_monthly_recalc = df_monthly_recalc.drop(columns=['_merge_key'])
    
    # Concatenate
    df_parent_merged = pd.concat([df_parent_keep, df_monthly_recalc], ignore_index=True)
    
    # Save
    df_parent_merged.to_csv(gold_parent_path, index=False)
    print(f"\n✓ Merged parent monthly fact: {gold_parent_path} now has {len(df_parent_merged)} rows")

print("\n" + "="*60)
print("INCREMENTAL LOAD COMPLETE")
print("="*60)
print(f"Bronze (full history): data/bronze_{data_source}.csv")
print(f"Silver (full history): data/silver_{data_source}.csv")
print(f"Gold daily (sb_fact): data/gold_sb_fact_{data_source}.csv")
print(f"Gold monthly (parent): data/gold_fact_{data_source}.csv")

## Cleanup

In [0]:
# Cleanup: Remove Bronze staging file (optional)
import os

staging_bronze_path = f"data/staging_bronze_{data_source}.csv"
if os.path.exists(staging_bronze_path):
    os.remove(staging_bronze_path)
    print(f"✓ Removed: {staging_bronze_path}")
else:
    print(f"Already removed: {staging_bronze_path}")

In [0]:
# Cleanup: Remove Silver staging file (optional)
staging_silver_path = f"data/staging_silver_{data_source}.csv"
if os.path.exists(staging_silver_path):
    os.remove(staging_silver_path)
    print(f"✓ Removed: {staging_silver_path}")
else:
    print(f"Already removed: {staging_silver_path}")